In [526]:
import numpy as np
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,accuracy_score,classification_report
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pandas

ASSIGMENT 2 Neural Network Dataset 1

In [527]:
# CHOOSE DATASET

# Binary classification dataset
data = datasets.load_diabetes(as_frame=True)

# Regression dataset
#data = datasets.fetch_openml(name="boston",version=1, as_frame=True) 

X = data.data.values
y = data.target.values 
X.shape

(442, 10)

In [528]:
#train test spliting
test_size=0.2
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=test_size, random_state=42)

In [529]:
# Standardize features
scaler=StandardScaler()
Xtr= scaler.fit_transform(Xtr)
Xte= scaler.transform(Xte)

This part applys standardized deviating for both training and thesting set.

In [530]:
class MLP(nn.Module):
    def __init__(self, input_size, output_size=1, dropout_prob=0.5):
        super(MLP, self).__init__()
        
        self.fc1 = nn.Linear(input_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 64)
        self.fc4 = nn.Linear(64, 64)
        self.out = nn.Linear(64, output_size)
        
        self.dropout = nn.Dropout(p=dropout_prob)
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        
        x = F.relu(self.fc3(x))
        x = self.dropout(x)
        
        x = F.relu(self.fc4(x))
        x = self.dropout(x)
        
        x = self.out(x)
        return x

Here is the creation of the neural network. First we create class MLP (multi layer perception = fully connected NN). Then is creation of the hidden layers, they are fully connected, because the input and output has the same size. Self dropout helps prevent overfitting. After that model goes through the layers and randomly does dropouts on each layer. Because there are only 4 hidden layers it is a shallow network.

In [531]:
num_epochs=100
lr=0.001
dropout=0.01
batch_size=64

Here we define number of cycles through the program, learning rate (step size), percentage of dropouts and number of samples it goes trough before the model updates its weights.

In [532]:
Xtr = torch.tensor(Xtr, dtype=torch.float32)
ytr = torch.tensor(ytr, dtype=torch.float32)
Xte = torch.tensor(Xte, dtype=torch.float32)
yte = torch.tensor(yte, dtype=torch.float32)

# Wrap Xtr and ytr into a dataset
train_dataset = TensorDataset(Xtr, ytr)

# Create DataLoader
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

Here we convert to PyTorch tensors in standard 32-bit floats, because PyTorch does not work with NumPy. Then we pair inputs with targets.

In [533]:
# Model, Loss, Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MLP(input_size=Xtr.shape[1], dropout_prob=dropout).to(device)
criterion = nn.BCEWithLogitsLoss()  # for binary classification
criterion = nn.MSELoss() #for regression
optimizer = optim.Adam(model.parameters(), lr=lr)

# Send data to device
Xtr, ytr = Xtr.to(device), ytr.to(device)
Xte, yte = Xte.to(device), yte.to(device)

Here code checks for availability of cuda, because I have cuda available, I needed to put in part of the porgram to transfer all the data to gpu. It also picks correct loss function.

In [534]:
# Training loop
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    for batch_x, batch_y in train_dataloader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        logits = model(batch_x)
        loss = criterion(logits, batch_y.view(-1, 1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_dataloader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

Epoch [1/100], Loss: 29622.7109
Epoch [2/100], Loss: 29677.5758
Epoch [3/100], Loss: 29390.6986
Epoch [4/100], Loss: 28807.3024
Epoch [5/100], Loss: 29018.1159
Epoch [6/100], Loss: 28931.0553
Epoch [7/100], Loss: 26665.4010
Epoch [8/100], Loss: 24685.1211
Epoch [9/100], Loss: 20604.1826
Epoch [10/100], Loss: 14269.8543
Epoch [11/100], Loss: 8149.2327
Epoch [12/100], Loss: 5852.4403
Epoch [13/100], Loss: 6101.0366
Epoch [14/100], Loss: 4782.0231
Epoch [15/100], Loss: 4495.6335
Epoch [16/100], Loss: 4255.8693
Epoch [17/100], Loss: 3904.4078
Epoch [18/100], Loss: 3890.1689
Epoch [19/100], Loss: 3755.9447
Epoch [20/100], Loss: 3694.8228
Epoch [21/100], Loss: 3542.5063
Epoch [22/100], Loss: 3524.4069
Epoch [23/100], Loss: 3447.0513
Epoch [24/100], Loss: 3320.7541
Epoch [25/100], Loss: 3303.2267
Epoch [26/100], Loss: 3434.7442
Epoch [27/100], Loss: 3289.2098
Epoch [28/100], Loss: 3208.3275
Epoch [29/100], Loss: 3216.6127
Epoch [30/100], Loss: 3215.7825
Epoch [31/100], Loss: 3215.6473
Epoch [

In here the real training is done. First we define how many epochs the code goes through, then we loop through batches. The steps of the loops are moving batch to gpu, getting prediction, computting loss, computting grasdients, upadting weights and printing loss per epoch.

In [535]:
y_pred=model(Xte)

# move both to cpu and convert to numpy
y_true_np = yte.detach().cpu().numpy()
y_pred_np = y_pred.detach().cpu().numpy()

#print(f'ACC:{accuracy_score(yte.detach().numpy(),y_pred.detach().numpy()>0.5)}') #classification
print(f'MSE:{mean_squared_error(y_true_np, y_pred_np)}') #regression

MSE:2816.077392578125


In this part we are getting back the tensor onto the cpu and converting it back to NumPy. Then we compute the MSE.

In comparason to the LS and to Anifs the result is worse, but that is probably due to my poor tuning. In general MLP is very flexibla and very usefull for larger dataset, but it needs proparly tunned hyperparametrs. The Anfis in comparasion has better interpretability and is better with small to medium sized data sets. Both of these are better then the LS which can capture accuratelly only linear interactions. 